# Tome 12, Chapter 11: Algebraic curves (Python companion)

Exercises first, solutions at the end. We count intersections with resultants and watch Bézout's number appear, resolve Cramer's paradox on an actual cubic over $\mathbb F_p$, measure when points fail to impose independent conditions on curves (the correction term of Riemann–Roch), count points of curves over finite fields as a preview of Chapter 13, and build an **algebraic-geometry code** from the Hermitian curve over $\mathbb F_{16}$: a code of length $64$ over a $16$-letter alphabet, which no Reed–Solomon code can match. Requires `cypari2`.

In [ ]:
import math, random, time
import cypari2
pari = cypari2.Pari()
pari.allocatemem(10**9, silent=True)
random.seed(11)

## Exercises

**P11.1 (★) Bézout by resultants.** For two projective curves $F,G$ of degrees $m,n$, make a random change of coordinates, set $z=1$, and compute the resultant $\operatorname{Res}_y(F,G)$, a polynomial in $x$. For generic coordinates its degree is $mn$ and the multiplicity of each root is the intersection number at the corresponding point. Check this on: the two parabolas of Figure 11.2, two circles, the cubic $y^2=x^3-x$ and its line at infinity, a nodal cubic and a line through the node, and two cubics.

**P11.2 (★★) Cramer's paradox.** Over $\mathbb F_p$, $p=10007$, take the smooth cubic $C_1:y^2=x^3+3x+7$ and eight random points on it. Show that the cubics through the eight points form a space of dimension $2$ (eight conditions on ten coefficients), find the ninth intersection point of $C_1$ with another cubic of this pencil, and check that *every* cubic through the eight points passes through it (Corollary 11.22(d)). Then check that nine random points of $C_1$ determine $C_1$.

**P11.3 (★★) Points that fail to impose independent conditions.** Let $C$ be the smooth quartic $x^4+y^4+z^4=0$ over $\mathbb F_p$, $p=10009$ ($g=3$, $W=E_1$). (a) By evaluating all forms of degree $d$ at many points of $C$, check $\dim V_d=4d-2$ for $d\ge2$ (Proposition 11.17). (b) For a set $D$ of $4$ points on a line, and for $4$ random points, compute how many independent conditions $D$ imposes on forms of degree $d=1,2,3$. Explain the difference with Riemann–Roch: the number of conditions is $\dim V_d-\ell(E_d-D)$.

**P11.4 (★) Points over $\mathbb F_p$ and the genus.** Count the points of random smooth cubics $y^2=x^3+ax+b$ and diagonal quartics $ax^4+by^4+cz^4$ over $\mathbb F_p$ for $p=101$ and $1009$, and of the Klein quartic $x^3y+y^3z+z^3x$. Compare $|N-p-1|$ with $2g\sqrt p$. (Chapter 13 proves the bound.)

**P11.5 (★★★) CS application: a Hermitian code.** Implement $\mathbb F_{16}=\mathbb F_2[t]/(t^4+t+1)$, find the $64$ affine points of $y^4+y=x^5$ (the Hermitian curve of Exercise 11.7 for $q=4$), and build the codes $\mathcal C_d$ of Proposition 11.23. Check that the dimension is $5d-5$ for $3\le d\le12$. For $d=6$, exhibit a codeword of weight exactly $34$ (so $\delta=34$), encode a random message, erase $33$ of the $64$ symbols and recover it. Compare with Reed–Solomon codes over $\mathbb F_{16}$.

## Solutions

### P11.1
**How to find it.** After a random projective change of coordinates, no intersection point is at infinity in the $y$-direction, and two intersection points have different $x$-coordinates; then the roots of $\operatorname{Res}_y$ are the $x$-coordinates of the points, and their multiplicities are the intersection numbers (Fulton, Section 5.1). A coordinate change that is not generic merges points, so we try several and keep the one with the most distinct roots.

In [ ]:
pari("tdeg(F) = poldegree(substvec(F, [x,y,z], [t*x,t*y,t*z]), t)")

def bezout(F, G, trials=40):
    m, n = int(pari(f"tdeg({F})")), int(pari(f"tdeg({G})"))
    best = None
    for _ in range(trials):
        M = [random.randint(-3, 3) for _ in range(9)]
        if pari.matdet(pari.matrix(3, 3, M)) == 0:
            continue
        v = ",".join(f"{M[3*i]}*x+{M[3*i+1]}*y+{M[3*i+2]}*z" for i in range(3))
        f = pari(f"subst(substvec({F},[x,y,z],[{v}]),z,1)")
        g = pari(f"subst(substvec({G},[x,y,z],[{v}]),z,1)")
        if pari.poldegree(f, 'y') != m or pari.poldegree(g, 'y') != n:
            continue
        R = pari.polresultant(f, g, 'y')
        if R == 0 or pari.poldegree(R) != m * n:
            continue
        fa = pari.factor(R)
        res = sorted((int(pari.poldegree(q)), int(e)) for q, e in zip(fa[0], fa[1]))
        if best is None or sum(d for d, e in res) > sum(d for d, e in best):
            best = res
    return m, n, best

for F, G, name in [("y*z-x^2", "y*z-2*x^2+z^2", "two parabolas"),
                   ("x^2+y^2-z^2", "x^2-2*x*z+y^2", "two circles"),
                   ("y^2*z-x^3+x*z^2", "z", "cubic and its line at infinity"),
                   ("y^2*z-x^2*(x+z)", "y-2*x", "nodal cubic, line through the node"),
                   ("y^2*z-x^3+x*z^2", "y^2*z-x^3-2*x^2*z+3*z^3", "two cubics")]:
    m, n, res = bezout(F, G)
    print(f"{name:36s} m*n = {m*n}; factors (degree, multiplicity): {res}; total {sum(d*e for d, e in res)}")

Each total is $mn$. The parabolas meet in two simple points and one double point (at infinity). The circles meet in two pairs of conjugate points: the real pair $(\frac12,\pm\frac{\sqrt3}2)$ and the circular points $(1:\pm i:0)$ (Exercise 11.1), each pair a quadratic factor. The line at infinity meets $y^2=x^3-x$ at a single point with multiplicity $3$: the point $(0:1:0)$ is a *flex*, which is why it is chosen as the origin of the group law in Chapter 12. A line through the node meets it with multiplicity $2$ there, plus one more point (Exercise 11.2). The two cubics $y^2=x^3-x$ and $y^2=x^3+2x^2-3$ share the flex at infinity with contact $5$, are tangent at $(1,0)$, and meet in a conjugate pair: $5+2+1+1=9$.

### P11.2
**How to find it.** A cubic has $10$ coefficients; vanishing at a point is one linear condition. The kernel of the $8\times10$ evaluation matrix is the pencil. The ninth point is found by intersecting $C_1$ with another member: the resultant in $x$ has degree $9$, eight of its roots are known, and the ninth must be in $\mathbb F_p$.

In [ ]:
p = 10007

def reduce_mod(rows, p):
    # reduced row echelon form; returns (rows, pivot columns)
    rows = [r[:] for r in rows]; rk = 0; piv = []
    for c in range(len(rows[0])):
        i = next((i for i in range(rk, len(rows)) if rows[i][c] % p), None)
        if i is None:
            continue
        rows[rk], rows[i] = rows[i], rows[rk]
        inv = pow(rows[rk][c], -1, p); rows[rk] = [v * inv % p for v in rows[rk]]
        for i in range(len(rows)):
            if i != rk and rows[i][c] % p:
                f = rows[i][c]; rows[i] = [(a - f * b) % p for a, b in zip(rows[i], rows[rk])]
        piv.append(c); rk += 1
    return rows[:rk], piv

def rank_mod(rows, p):
    return len(reduce_mod(rows, p)[1]) if rows else 0

def nullspace_mod(rows, p):
    R, piv = reduce_mod(rows, p); ncol = len(rows[0]); basis = []
    for free in (c for c in range(ncol) if c not in piv):
        v = [0] * ncol; v[free] = 1
        for r, c in zip(R, piv):
            v[c] = -r[free] % p
        basis.append(v)
    return basis

def monomials(d):
    return [(i, j, d - i - j) for i in range(d + 1) for j in range(d + 1 - i)]

def evalmono(P, d, p):
    x, y, z = P
    return [pow(x, i, p) * pow(y, j, p) * pow(z, k, p) % p for i, j, k in monomials(d)]

def form(v, d):
    return "+".join(f"({c})*x^{i}*y^{j}*z^{k}" for c, (i, j, k) in zip(v, monomials(d)) if c) or "0"

a, b = 3, 7
def points_on_cubic(n):
    pts = set()
    while len(pts) < n:
        x = random.randrange(p); r = (x**3 + a*x + b) % p
        if pow(r, (p - 1) // 2, p) == 1:
            y = int(pari.sqrt(pari.Mod(r, p)).lift())
            pts.add((x, random.choice([y, p - y]), 1))
    return sorted(pts)

P8 = points_on_cubic(8)
pencil = nullspace_mod([evalmono(P, 3, p) for P in P8], p)
print("dimension of the space of cubics through the 8 points:", len(pencil))

C1 = pari(f"y^2*z-x^3-{a}*x*z^2-{b}*z^3")
c1vec = [int(pari.polcoef(pari.polcoef(pari.polcoef(C1, i, 'x'), j, 'y'), k, 'z')) % p for i, j, k in monomials(3)]
C2v = next(v for v in pencil if rank_mod([v, c1vec], p) == 2)       # a member not proportional to C1
f1 = pari.subst(C1, 'z', 1) * pari.Mod(1, p)
f2 = pari.subst(pari(form(C2v, 3)), 'z', 1) * pari.Mod(1, p)
R = pari.polresultant(f1, f2, 'y')
xs = [int(r.lift()) for r in pari.polrootsmod(pari.lift(R), p)]
x9 = [x for x in xs if x not in {P[0] for P in P8}]
print("degree of the resultant:", pari.poldegree(R), "; new root(s):", x9)
x9 = x9[0]
g = pari.gcd(pari.subst(f1, 'x', x9), pari.subst(f2, 'x', x9))          # linear in y
y9 = int((-pari.polcoef(g, 0) / pari.polcoef(g, 1)).lift())
P9 = (x9, y9, 1)
print("ninth point:", P9)
ok = all(sum(c * e for c, e in zip(v, evalmono(P9, 3, p))) % p == 0 for v in pencil)
lam, mu = random.randrange(1, p), random.randrange(1, p)
rnd = [(lam * u + mu * w) % p for u, w in zip(*pencil)]
print("every cubic of the pencil vanishes at P9:", ok, "; a random member too:",
      sum(c * e for c, e in zip(rnd, evalmono(P9, 3, p))) % p == 0)
print("cubics through 9 random points of C1:", len(nullspace_mod([evalmono(P, 3, p) for P in points_on_cubic(9)], p)))

Eight points cut out a pencil (dimension $2$) and the ninth base point $P_9$ comes for free: that is Cramer's paradox and its resolution. Nine random points of $C_1$ leave a space of dimension $1$, that is $C_1$ alone; nine points *in general position* in the plane determine a unique cubic, but the nine intersection points of two cubics are not in general position. In Chapter 12, $P_9$ will be computed directly: with $O$ the point at infinity and the group law, $P_9=-(P_1+\dots+P_8)$.

### P11.3
**How to find it.** $\dim V_d$ is the rank of the evaluation matrix at more than $dn$ points (Proposition 11.23's argument: a form of degree $d$ not divisible by $F$ has at most $dn$ zeros on $C$). The number of conditions imposed by a set $D$ of points is the rank of the evaluation matrix restricted to $V_d$, which is $\dim V_d-\ell(E_d-D)$ by Lemma 11.14.

In [ ]:
p = 10009          # p = 1 mod 4: the quartic has plenty of points
def quartic_points(n):
    pts = set()
    while len(pts) < n:
        x = random.randrange(p)
        for y in pari.polrootsmod(pari(f"y^4+{x}^4+1"), p):
            pts.add((x, int(y.lift()), 1))
    return sorted(pts)[:n]

PTS = quartic_points(60)
for d in range(1, 7):
    rk = rank_mod([evalmono(P, d, p) for P in PTS], p)
    print(f"d={d}: dim V_d = {rk}   (formula: {(d+1)*(d+2)//2 if d < 2 else 4*d-2})")

# four points on a line: y = c meets C in the roots of x^4 = -(c^4+1)
c = next(c for c in range(1, p) if len(pari.polrootsmod(pari(f"x^4+{c}^4+1"), p)) == 4)
line = [(int(r.lift()), c, 1) for r in pari.polrootsmod(pari(f"x^4+{c}^4+1"), p)]
general = quartic_points(400)[::97][:4]
def conditions(D, d):
    # rank of evaluation at D on V_d = forms of degree d modulo F
    basis = reduce_mod([evalmono(P, d, p) for P in PTS], p)[1]          # monomials independent on C
    return rank_mod([[evalmono(P, d, p)[i] for i in basis] for P in D], p)
for d in (1, 2, 3):
    print(f"d={d}: 4 collinear points impose {conditions(line, d)} conditions, 4 random points impose {conditions(general, d)}")

The four collinear points form a line section $D=\operatorname{div}(M)=W$ (on a quartic, $W=E_1$), so $\ell(E_d-D)=\ell(E_{d-1})=\dim V_{d-1}$ and they impose $\dim V_d-\dim V_{d-1}=2,3,4$ conditions for $d=1,2,3$; for random points the count is $\min(4,\dim V_d)=3,4,4$. Riemann–Roch for $E_d-D$ reads $\ell(E_d-D)=4d-6+\ell(W-E_d+D)$ with $W-E_d+D\equiv E_{2-d}$: for $d=2$ the correction $\ell(E_0)=1$ is exactly the missing condition, and for $d=3$ it is $\ell(E_{-1})=0$, so collinear points behave like general ones. Geometrically: a conic through three collinear points contains their line (Bézout, $3>2$), hence passes through the fourth point for free.

### P11.4
**How to find it.** For each $x\in\mathbb F_p$ count the roots of $F(x,y,1)$ in $y$, then add the points on $z=0$.

In [ ]:
def count(F, p):
    F = pari(F); N = 0
    for x in range(p):
        f = pari.substvec(F, ['x', 'z'], [x, 1]) * pari.Mod(1, p)
        if f == 0:
            N += p
        elif pari.poldegree(f, 'y') > 0:
            N += len(pari.polrootsmod(pari.lift(f), p))
    g = pari.substvec(F, ['y', 'z'], [1, 0]) * pari.Mod(1, p)            # points (x:1:0)
    N += p if g == 0 else (len(pari.polrootsmod(pari.lift(g), p)) if pari.poldegree(g, 'x') > 0 else 0)
    N += pari.substvec(F, ['x', 'y', 'z'], [1, 0, 0]) % p == 0          # the point (1:0:0)
    return N

random.seed(3)
for p in (101, 1009):
    worst = {1: 0, 3: 0}
    for _ in range(20):
        a, b = random.randrange(p), random.randrange(p)
        if (4*a**3 + 27*b*b) % p:
            N = count(f"y^2*z-x^3-{a}*x*z^2-{b}*z^3", p)
            worst[1] = max(worst[1], abs(N - p - 1) / (2 * math.sqrt(p)))
    for _ in range(8):
        a, b, c = [random.randrange(1, p) for _ in range(3)]
        N = count(f"{a}*x^4+{b}*y^4+{c}*z^4", p)
        worst[3] = max(worst[3], abs(N - p - 1) / (6 * math.sqrt(p)))
    N = count("x^3*y+y^3*z+z^3*x", p)
    print(f"p={p}: max |N-p-1|/(2g sqrt p): cubics {worst[1]:.3f}, quartics {worst[3]:.3f};  Klein quartic N={N}")

All the ratios are below $1$. For cubics the worst case gets close to $1$ (the values $N-p-1$ fill the interval $[-2\sqrt p,2\sqrt p]$, Chapter 13 and the Sato–Tate law), while the Klein quartic has $N=p+1$ exactly for $p=101$ and nearly so for $p=1009$. The bound $|N-p-1|\le2g\sqrt p$ is the Riemann hypothesis for curves over finite fields, proved by Weil in 1948; Chapter 13 proves it for smooth plane curves.

### P11.5
**How to find it.** $\mathbb F_{16}$ by log and antilog tables. In $V_d$, the relation $x^5=y^4+y$ (the top-degree part of $F_*$ is $x^5$) lets one reduce every monomial to $x^iy^j$ with $i\le4$, $i+j\le d$, and these form a basis of $V_d$ ($5d-5$ of them for $d\ge3$). Codewords of weight $34$ come from products of $6$ lines meeting the curve in $5$ affine points each: the horizontal lines $y=c$ with $c^4+c\ne0$, since $x\mapsto x^5$ is $5$-to-$1$ from $\mathbb F_{16}^\times$ onto $\mathbb F_4^\times$. Erasure decoding is a linear system on the $31$ surviving coordinates, which determine the message because a non-zero codeword has at most $30$ zeros.

In [ ]:
EXP = [0] * 30; LOG = [0] * 16; v = 1
for i in range(15):
    EXP[i] = v; LOG[v] = i; v <<= 1
    if v & 16:
        v ^= 0b10011                     # t^4 = t + 1
for i in range(15, 30):
    EXP[i] = EXP[i - 15]
def mul(a, b): return 0 if a == 0 or b == 0 else EXP[LOG[a] + LOG[b]]
def inv(a): return EXP[(15 - LOG[a]) % 15]
def pw(a, e): return 1 if e == 0 else (0 if a == 0 else EXP[LOG[a] * e % 15])

pts = [(x, y) for x in range(16) for y in range(16) if pw(y, 4) ^ y == pw(x, 5)]
print("affine points of y^4 + y = x^5 over F_16:", len(pts))

def rref_gf(rows):
    rows = [r[:] for r in rows]; rk = 0; piv = []
    for c in range(len(rows[0])):
        i = next((i for i in range(rk, len(rows)) if rows[i][c]), None)
        if i is None:
            continue
        rows[rk], rows[i] = rows[i], rows[rk]
        iv = inv(rows[rk][c]); rows[rk] = [mul(x, iv) for x in rows[rk]]
        for i in range(len(rows)):
            if i != rk and rows[i][c]:
                f = rows[i][c]; rows[i] = [a ^ mul(f, b) for a, b in zip(rows[i], rows[rk])]
        piv.append(c); rk += 1
    return rows[:rk], piv

def generator(d):
    mons = [(i, j) for i in range(min(d, 4) + 1) for j in range(d - i + 1)]
    return mons, [[mul(pw(x, i), pw(y, j)) for (x, y) in pts] for (i, j) in mons]

for d in range(3, 13):
    mons, Gm = generator(d)
    print(f"d={d:2d}: [64, {len(rref_gf(Gm)[1])}, >= {64 - 5*d}]   (5d-5 = {5*d - 5})")

In [ ]:
d = 6
mons, Gm = generator(d)
k = len(mons)
# a codeword of weight 34: the product of 6 horizontal lines y = c with c^4 + c != 0
cs = [c for c in range(16) if pw(c, 4) ^ c][:6]
word = []
for (x, y) in pts:
    v = 1
    for c in cs:
        v = mul(v, y ^ c)
    word.append(v)
print("weight of prod (y - c):", sum(1 for w in word if w), " so the minimum distance is exactly 34")

# encode, erase 33 symbols, decode
msg = [random.randrange(16) for _ in range(k)]
cw = [0] * 64
for m, row in zip(msg, Gm):
    for i in range(64):
        cw[i] ^= mul(m, row[i])
keep = sorted(random.sample(range(64), 64 - 33))
# solve msg * G[:, keep] = cw[keep]: transpose to rows = equations
A = [[Gm[r][i] for r in range(k)] + [cw[i]] for i in keep]
R, piv = rref_gf(A)
decoded = [0] * k
for row, c in zip(R, piv):
    decoded[c] = row[-1]
print("pivots:", len(piv), "; message recovered from 31 of 64 symbols:", decoded == msg)

The code $\mathcal C_6$ is a $[64,25,34]$ code over $\mathbb F_{16}$: $\kappa+\delta=59=N+1-g$, six below the Singleton bound $N+1=65$, and the six is the genus. A Reed–Solomon code over $\mathbb F_{16}$ meets Singleton exactly but has length at most $16$ (or $17$ with the point at infinity): to encode $64$ symbols one would need $\mathbb F_{64}$ or larger, with larger symbols and slower arithmetic. This trade (a little distance for a lot of length) is the reason codes from curves exist; the Hermitian curve is the favourite because it has the most points allowed by its genus (Exercise 11.7(c)).